# #189 パスの確認（CPU、GPU 0、提出しない）
診断ノート（casmi26-lb487-btables）の凍結セル 32 の表の決め方をそのまま実行し、2 本の npz の sha256 と名前の重なりを確かめる。

In [ ]:
SLUG = 'casmi26-btables-pathcheck'
KSRC_NAME = 'casmi26-engine2-build-b-run1'
SIM_REL = 'own/own_sim_rank_rows.npz'
SIM_SHA = '9ee9011314e158ef410e499368137e96aec8eafc13d3999594cfa70b3650d013'
SIM_SIZE = 90404186
RT_REL = 'own/own_rank_train.npz'
RT_SHA = '74c0a37545d8c3cc578ef14d24ac79cfc1c9cbdae7eb65fe8c76ccf4633bf269'
RT_SIZE = 63410966
LOOKUP = ['coco_fp.npy', 'coco_meta.pkl', 'fp_bits.npy', 'bio_fp.npy', 'bio_meta.pkl', 'pubchemlite_fp.npy', 'pubchemlite_meta.pkl', 'pool_meta.parquet', 'pc_smiles.npy', 'test.parquet', 'train.parquet', 'sample_submission.csv', 'pool_lsid.npy', 'pool_lpmid.npy', 'pool_popularity.csv', 'final_model.pt', 'dreams_np_mass.npy', 'ext_meta.parquet', 'ext_mz.npy', 'ext_in.npy', 'ext_mass.parquet', 'spectra.parquet', 'candidates_lotus.parquet', 'engine.py', 'MANIFEST.json', 'ranker_0.pkl', 'ice_runner.py', 'gl_runner.py', 'fpnet_full1.pt', 'rank_train.npz', 'sim_rank_rows_nofp.npz']
LOADER = '# ── #189（#158 (B) の診断）: Engine 2 の 2 表だけを自前表（b_run1 version 3）に。推論の FP は元のまま ──\nimport glob as _e2g, json as _e2j, hashlib as _e2h\n_E2_KSRC = \'casmi26-engine2-build-b-run1\'\ndef _e2own(rel, sha):\n    # kernel_sources の b_run1 の下にあり rel で終わるファイルがちょうど 1 つ。sha256 が違えば Engine 2 を起動する前に止める\n    _h = sorted(p for p in _e2g.glob(\'/kaggle/input/**/\' + rel, recursive=True)\n                if _E2_KSRC in p.split(\'/\')[:-1] and p.endswith(\'/\' + rel))\n    if len(_h) != 1:\n        raise RuntimeError((\'#189 own table not found exactly once under \' + _E2_KSRC, rel, _h))\n    _s = _e2h.sha256(open(_h[0], \'rb\').read()).hexdigest()\n    if _s != sha:\n        raise RuntimeError((\'#189 own table sha256 mismatch\', _h[0], _s, sha))\n    return _h[0]\n# #108 09:19Z の 2: 再実行では自前表の確認の失敗も Engine 2 の失敗と同じく v4g のリストへの戻りにする。commit では止める\n_E2_FAIL = None\ntry:\n    _E2_SIM = _e2own(\'own/own_sim_rank_rows.npz\', \'9ee9011314e158ef410e499368137e96aec8eafc13d3999594cfa70b3650d013\')\n    _E2_RT = _e2own(\'own/own_rank_train.npz\', \'74c0a37545d8c3cc578ef14d24ac79cfc1c9cbdae7eb65fe8c76ccf4633bf269\')\nexcept Exception as _e2x:\n    if not IS_RERUN:\n        raise\n    _E2_FAIL = (\'own tables\', repr(_e2x)); _E2_SIM = _E2_RT = None\n_E2_OWN = dict(sim=_E2_SIM, sim_sha=\'9ee9011314e158ef410e499368137e96aec8eafc13d3999594cfa70b3650d013\', rt=_E2_RT, rt_sha=\'74c0a37545d8c3cc578ef14d24ac79cfc1c9cbdae7eb65fe8c76ccf4633bf269\')\nopen(os.path.join(ENG_DIR, \'eng_runner.py\'), \'w\').write(\'"""Our engine (two-ranker, BIO + AFIX) -> {molecule_id: {smiles: [...], keys: [...]}} (blend order, top 40).\\n#189 (#158 (B)): same as the original runner, but the two Engine 2 tables are read from explicit paths inside the\\nkernel source taichiiiii/casmi26-engine2-build-b-run1 (sha256 checked). FP inference is unchanged (fp-models-v2)."""\\nimport os, sys, json, time\\nimport numpy as np\\nimport hashlib\\nKSRC_NAME = \\\'casmi26-engine2-build-b-run1\\\'\\nRP = \\\'/kaggle/working/engine2_btables_report.json\\\'\\n\\n\\ndef verify_own_tables(own):\\n    """own: {\\\'sim\\\': path, \\\'sim_sha\\\': hex, \\\'rt\\\': path, \\\'rt_sha\\\': hex} -> {\\\'sim\\\': sha, \\\'rt\\\': sha}. Raises on any mismatch."""\\n    sha = {}\\n    for k in (\\\'sim\\\', \\\'rt\\\'):\\n        p = own[k]\\n        if KSRC_NAME not in os.path.normpath(p).split(os.sep)[:-1] or not os.path.basename(p).startswith(\\\'own_\\\'):\\n            raise ValueError((\\\'#189 own table outside \\\' + KSRC_NAME, k, p))\\n        h = hashlib.sha256(open(p, \\\'rb\\\').read()).hexdigest()\\n        if h != own[k + \\\'_sha\\\']:\\n            raise ValueError((\\\'#189 own table sha256 mismatch\\\', k, p, h, own[k + \\\'_sha\\\']))\\n        sha[k] = h\\n    return sha\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    here = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\\n    test, train, sample, out = sys.argv[1:5]\\n    OWN = json.loads(os.environ[\\\'E2_OWN\\\'])\\n    REP = dict(status=\\\'started\\\', own=OWN)\\n    json.dump(REP, open(RP, \\\'w\\\'), indent=1)\\n    import casmi_engine as E\\n    _orig_find = E.find\\n    def _find(n):                                  # v4g ships its own fp_bits.npy (10,226 bits): take prvsiyan\\\'s 6,930\\n        if n == \\\'fp_bits.npy\\\':\\n            return os.path.join(os.path.dirname(_orig_find(\\\'coco_fp.npy\\\')), \\\'fp_bits.npy\\\')\\n        assert \\\'rank_train\\\' not in n and \\\'sim_rank_rows\\\' not in n, (\\\'original Engine 2 table requested\\\', n)\\n        return _orig_find(n)\\n    E.find = _find\\n    E.RANK.W_A = (0.35, 0.55); E.RANK.SEEDS = (0, 1); E.CFG.N_ANALOG = 200\\n    import glob\\n    fp_models = sorted(p for p in glob.glob(\\\'/kaggle/input/**/fp_*.pt\\\', recursive=True) if \\\'casmi26-fp-models-v2\\\' in p)\\n    print(\\\'engine fp models\\\', fp_models, flush=True)\\n    sha = verify_own_tables(OWN)\\n    REP.update(tables_read={k: OWN[k] for k in (\\\'sim\\\', \\\'rt\\\')}, tables_sha256=sha, fp_models=fp_models)\\n    T0 = time.time()\\n    subs, recs, S = E.main(test, train, sample, OWN[\\\'sim\\\'], OWN[\\\'rt\\\'], fp_models,\\n                           workers=os.cpu_count(), w_pv=0.88)\\n    bl = E.blend_scores(S[\\\'pv\\\'], S[\\\'ours\\\'], 0.88)\\n    res = {}\\n    for r in recs:\\n        mid = r[\\\'mid\\\']\\n        if mid not in bl:\\n            continue\\n        order = np.argsort(-bl[mid], kind=\\\'mergesort\\\')[:80]\\n        smis, keys, seen = [], [], set()\\n        for c in r[\\\'cand\\\'][order]:\\n            s = E.POOL[\\\'smiles\\\'][c]; k = E.canon_key(s)\\n            if k is None or k in seen:\\n                continue\\n            seen.add(k); smis.append(s); keys.append(k)\\n            if len(smis) >= 40:\\n                break\\n        res[str(mid)] = dict(smiles=smis, keys=keys)\\n    json.dump(res, open(out, \\\'w\\\'))\\n    print(\\\'engine lists\\\', len(res), f\\\'{time.time()-T0:.0f}s\\\', flush=True)\\n    REP.update(status=\\\'ok\\\', molecules_listed=len(res), engine_secs=round(time.time() - T0, 1))\\n    json.dump(REP, open(RP, \\\'w\\\'), indent=1, default=str)\\n\')\nos.environ[\'E2_OWN\'] = _e2j.dumps(_E2_OWN)\n_E2_EXC = None   # 下の except で Engine 2 の例外の repr を残す\nrr = None        # subprocess が終わらなかった（時間切れ・起動の失敗）ときは None のまま\nprint(\'Engine 2 own tables (#189):\', _E2_OWN, flush=True)\n'

import os, sys, json, glob, hashlib, time
T0 = time.time()
OUT = {'slug': SLUG, 'status': 'started'}
ENG_DIR = '/kaggle/working/eng'; os.makedirs(ENG_DIR, exist_ok=True)
IS_RERUN = False
# ---- 診断ノートの凍結セル 32 の表の決め方（そのまま） ----
exec(compile(LOADER, 'frozen_cell_32_loader', 'exec'))
OUT['own'] = _E2_OWN
fails = []
for k, rel, sha, size in (('sim', SIM_REL, SIM_SHA, SIM_SIZE), ('rt', RT_REL, RT_SHA, RT_SIZE)):
    p = _E2_OWN[k]
    got = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    OUT[k] = dict(path=p, sha256=got, size=os.path.getsize(p), sha_ok=got == sha, size_ok=os.path.getsize(p) == size)
    if got != sha or os.path.getsize(p) != size:
        fails.append(('table', k, p, got, os.path.getsize(p)))
# ---- runner 側の照合（書かれた eng_runner.py の verify_own_tables） ----
rg = {'__name__': 'eng_runner_check'}
exec(compile(open(os.path.join(ENG_DIR, 'eng_runner.py')).read(), 'eng_runner.py', 'exec'), rg)
OUT['runner_verify'] = rg['verify_own_tables'](_E2_OWN)
# ---- 名前の重なり ----
by_name = {}
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    for f in files:
        by_name.setdefault(f, []).append(os.path.join(root, f))
def in_b(p):
    return KSRC_NAME in p.split('/')[:-1]
b_root = sorted({p.split('/' + KSRC_NAME + '/')[0] + '/' + KSRC_NAME for ps in by_name.values() for p in ps if in_b(p)})
OUT['b_run1_roots'] = b_root
if len(b_root) != 1:
    fails.append(('b_run1 root not unique', b_root))
b_names = sorted({os.path.basename(p) for ps in by_name.values() for p in ps if in_b(p)})
OUT['b_run1_files'] = sum(1 for ps in by_name.values() for p in ps if in_b(p))
overlap = {}
for n in b_names:
    others = [p for p in by_name[n] if not in_b(p)]
    if others:
        first = sorted(by_name[n], key=len)[0]
        overlap[n] = dict(others=len(others), first_is_b_run1=in_b(first))
        if in_b(first):
            fails.append(('find() first hit moves into b_run1', n, first))
OUT['name_overlap'] = overlap
lk = {}
for n in LOOKUP:
    hits = by_name.get(n, [])
    lk[n] = dict(total=len(hits), in_b_run1=sum(in_b(p) for p in hits))
    if any(in_b(p) for p in hits):
        fails.append(('lookup name inside b_run1', n))
OUT['lookup'] = lk
fp_pt = sorted(p for p in glob.glob('/kaggle/input/**/fp_*.pt', recursive=True) if 'casmi26-fp-models-v2' in p)
OUT['engine_fp_models'] = fp_pt
if any(in_b(p) for p in glob.glob('/kaggle/input/**/fp_*.pt', recursive=True)):
    fails.append(('fp_*.pt inside b_run1',))
OUT['input_top'] = sorted(os.listdir('/kaggle/input'))
OUT['fails'] = fails
OUT['status'] = 'ok' if not fails else 'fail'
OUT['secs'] = round(time.time() - T0, 1)
json.dump(OUT, open('/kaggle/working/pathcheck.json', 'w'), indent=1, default=str)
print('PATHCHECK ' + json.dumps(dict(status=OUT['status'], sim=OUT['sim'], rt=OUT['rt'], fails=fails,
                                     b_run1_roots=b_root, name_overlap=overlap), default=str), flush=True)
assert not fails, fails
